In [2]:
import SHAConstants
from typing import List

> ## Utils functions

In [3]:
def hex_f(number, name, end="\n"):
    hex__ = format(number, '08x')
    if end == "\n":
        print(f"[+] > {name} : {" ".join([hex__[i:i+2] for i in range(0, len(hex__), 2)])}", end=end)
    else:
        print(f"({" ".join([hex__[i:i+2] for i in range(0, len(hex__), 2)])})", end=end)


def print_hex(hex_list, k = 16):
    """
        print hex list
    """
    for i in range(0, len(hex_list), k):
        print(" ".join(hex_list[i:i+k]))
    print()


def hex2dec(hex_string):
    """
        Converts hex string to décimal.
    """
    return int(hex_string, 16)


def dec2hex(number, n_nibbles):
    """
        Converts dec string to hex.
    """
    return format(number, f"0{n_nibbles}x")



def _xor(number1: int, number2: int) -> int:
    """
        Performs xor operations between numbers
    """
    return number1 ^ number2


def _and(number1: int, number2: int) -> int:
    """
        Performs and operations on a binary strings
    """
    return number1 & number2


def _not(number: int, n_bits: int = 32) -> int:
    """
        Performs NOT operations on a binary string.
    """
    # mask of n_bits
    mask = (1 << n_bits) - 1

    return ~number & mask



def right_shift(number : int, n_pos:int) ->int:
    """
        Shifts by 'n_pos' positions, padding with zeros.
    """
    # Shift right by n_pos positions
    return number >> n_pos


def right_rotate(number: int, n_pos:int, n_bits: int = 32) -> int:
    """
        Rotates a binary number to the right by 'n_pos' positions, preserving the bit length.
    """
    bin_chain = format(number, f'0{n_bits}b')  # Convert to binary with leading zeros
    k = n_bits - n_pos % n_bits                # Compute the split index
    rotated = bin_chain[k:] + bin_chain[:k]    # Perform the rotation
    return int(rotated, 2)                     # Convert back to integer


def sum_mod32(*numbers: int) -> int:
    """
        Sums multiple integers (32-bit), performing the addition modulo 2^32
    """
    for n in numbers:
        assert 0 <= n < 2**32, "Each number must fit in 32 bits (unsigned integer)"
    
    return sum(numbers) % (2**32)



> ## Padding

In [4]:
def padd_message(message: str) -> List[str]:
    """
        Pads a message according to the SHA-256 algorithm.
    """
    # converts message to hexadecimal
    hex_message = [format(byte, '02x') for byte in message.encode('utf-8')]
    L = len(hex_message) * 2    # len of hex message

    # hex message length calculation
    original_length = len(message.encode('utf-8')) * 8

    # Add the bit '1' (which is 0x80 in hexadecimal)
    hex_message.append('80')


    # Calculate the number of padding zeros needed to make the length a multiple of 512 bits
    L = len(hex_message)
    K = (64 - (L + 8) % 64) % 64  # 64 because we work with hexadecimal (2 hex digit = 8 bits)

    # Add K bits of '0' and L under 64 bits to bin_message
    hex_message += ['00'] * K

    # Add the length of the original message (in bits) in 64 bits (16 hex characters)
    length_in_hex = format(original_length, '016x')  # Length in hex (64 bits)
    hex_message += [length_in_hex[i:i+2] for i in range(0, len(length_in_hex), 2)]

    #
    return hex_message

> ## Merkle-Damgard

In [5]:
def merkel_damgard(hex_message:List[str]):
    """
        Applies the Merkle-Damgård construction for SHA-256 hashing.
    """
    # Length of the padded message
    l = len(hex_message)

    # slice into 512-bit blocks (64 hex bits)                                                        
    chuncks = [hex_message[i:i+64] for i in range(0, l, 64)]

    # IV eight 32 bit words
    h = SHAConstants.IV

    # Iterate through each 512-bit block of the message (64 hex blocks)
    for chunck_64hex in chuncks:
        # Call the one-way compression function with the current block and the current hash value
        h = one_way_compression(chunck_64hex, h)
    return h

> ## One-way compression function

In [6]:
def one_way_compression(chunck_64hex, IV_words) -> List[int]:
    # -------------------------------------------------------------------
    ## STEP 1 : CREATION OF 16 WORDS of 32bits (8 hex nibbles)

    # i need to choose 4 elements in the hex chunk (XX XX XX XX) -> (4*2 + 4*2 + 4*2 + 4*2) -> 32 bits = (8 hex nibbles)
    W = [
        "".join(chunck_64hex[i:i+4]) for i in range(0, len(chunck_64hex), 4)
    ]

    # conversion to list of int
    W = [int(x, 16) for x in W]
    
    # other words constructions
    for i in range(16, 64):
        # s0 building
        s0 = _xor(
            right_rotate(W[i - 15], 7),
            _xor(
                right_rotate(W[i - 15], 18),
                right_shift(W[i - 15], 3)
            )
        )

        
        # s1 building
        s1 = _xor(
            right_rotate(W[i - 2], 17),
            _xor(
                right_rotate(W[i - 2], 19),
                right_shift(W[i - 2], 10)
            )
        )

        
        # Wi computation and add to W
        W_i = sum_mod32(W[i - 16], s0, W[i - 7], s1)
        #
        W.append(W_i)
    
    # -------------------------------------------------------------------
    ## STEP 2 : 64 iterations of compression process
    a, b, c, d, e, f, g, h = IV_words

    # loop
    for i in range(64):
        # X1 building
        X_1 = _xor(
            right_rotate(e, 6),
            _xor(
                right_rotate(e, 11),
                right_rotate(e, 25)
            )
        )

        # CH building
        CH = _xor(
            _and(e, f),
            _and(
                _not(e),
                g
            )
        )

        # X2 building
        X_2 = _xor(
            right_rotate(a, 2),
            _xor(
                right_rotate(a, 13),
                right_rotate(a, 22)
            )
        )

        # MAJ building
        MAJ = _xor(
            _and(a, b),
            _xor(
                _and(a, c),
                _and(b, c)
            )
        )

        # temp1 computing
        K_i = SHAConstants.K[i]
        temp1 = sum_mod32(h, X_1, CH, K_i, W[i])

        # temp2 computing
        temp2 = sum_mod32(X_2, MAJ)

        # update and computing of other variables
        h, g, f = g, f, e
        e = sum_mod32(d, temp1)
        d, c, b = c, b, a
        a = sum_mod32(temp1, temp2)
        
    
    # -------------------------------------------------------------------
    ## STEP 3 : Results combination
    a_to_h = (a, b, c, d, e, f, g, h)

    # new IV
    new_IV = [sum_mod32(h_i, x) for h_i, x in zip(IV_words, a_to_h)]
    
    return new_IV

> ### our SHA

In [7]:
def SHA256(message: str) -> (List[str], str):
    # hex padding and printing
    hex_pad = padd_message(message)
    
    # slicing + compression
    merkel_dam = merkel_damgard(hex_pad)

    # conversion to hex
    hex_merkel_dam = [dec2hex(dec, n_nibbles=8) for dec in merkel_dam]

    return hex_pad, "".join(hex_merkel_dam)

In [8]:
for message in [SHAConstants.ex1, SHAConstants.ex2, SHAConstants.ex3]:
    print("=" * 70)
    print("=" * 70)
    
    # printing message
    print(f"[+] > message: {message}")

    # hash operation
    hex_pad, hashed_message = SHA256(message)
    
    print(f"\n[+] > Hexadecimal representation of the message:\n")
    print_hex(hex_pad)
    print(f"[+] > hashed message: {hashed_message} \n")


[+] > message: 

[+] > Hexadecimal representation of the message:

80 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00

[+] > hashed message: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 

[+] > message: Welcome to Wrestlemania!

[+] > Hexadecimal representation of the message:

57 65 6c 63 6f 6d 65 20 74 6f 20 57 72 65 73 74
6c 65 6d 61 6e 69 61 21 80 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 c0

[+] > hashed message: 70eeb26f0052ebe0041e58d221e954c575f32a979cefdae7b761969e33b7934f 

[+] > message: I will travel across the land
Searching far and wide
Teach Pokemon to understand
The power that's inside

[+] > Hexadecimal representation of the message:

49 20 77 69 6c 6c 20 74 72 61 76 65 6c 20 61 63
72 6f 73 73 20 74 68 65 20 6c 61 6e 64 0a 53 65
61 72 63 68

> # PROOF OF WORK

In [10]:
def proof_of_work(lastname, sha256_function, nonce_start = 0, show_evolution = False):
    nonce = nonce_start
    while True:
        # Construct the message by appending the nonce to the lastname
        message = lastname + str(nonce)
        
        # Compute the SHA-256 hash of the message
        _, digest = sha256_function(message)
        if show_evolution:
          print(f"[+] > {nonce} > {digest[-5:]}")
        
        # Check if the last 5 hexadecimal digits of the digest are '00000' (i.e. 20 zero bits)
        if digest.endswith("00000"):
            print(f"Message: {message}")
            print(f"Nonce: {nonce}")
            print(f"Digest: {digest}")
            return nonce, digest
        
        # Increment nonce and try again
        nonce += 1

In [ ]:
# call to function : Test 1
lastname = "ALEXA"
nonce, digest = proof_of_work(lastname, SHA256, nonce_start=90000, show_evolution=True)